In [119]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler, OrdinalEncoder
from sklearn.feature_selection import chi2, mutual_info_classif
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
from sklearn.naive_bayes import GaussianNB
from sklearn.metrics import accuracy_score
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import LabelEncoder
import re



In [120]:
# -----------------------------
# Load dataset
# -----------------------------
df = pd.read_csv("D:\\2nd Sem\\ML Pipelines\\The Titanic dataset.csv")

# --------------------------------
# Drop useless columns
# --------------------------------
df = df.drop(columns=["sn", "name", "date"])


In [121]:
df.head()


,pclass,survived,gender,age,family,fare,embarked
0,3,0,male,42.0,0.0,7.55,NaN
1,3,0,male,42.0,0.0,7.55,NaN
2,3,0,male,NaN,2.0,20.25,S
3,2,0,NaN,NaN,2.0,**,S
4,3,1,female,35.0,2.0,20.25,S


In [122]:
def clean_dataframe_text(df, columns):
    for col in columns:
        # Step 1: standardize missing markers
        df[col] = df[col].replace(["?", "**", "NA", "N/A", ""], np.nan)

        # Step 2: clean remaining text
        df[col] = (
            df[col]
            .astype(str)
            .str.replace(r'[^\x00-\x7F]+', ' ', regex=True)
            .str.strip()
            .str.replace(r'\s+', ' ', regex=True)
            .str.replace(r'[~!@#$%^&*()_+=`{}|\[\]\\:";\'<>?,./-]', '', regex=True)
        )

    return df
df = clean_dataframe_text(df, ['pclass', 'survived', 'gender', 'age', 'family', 'fare', 'embarked'])


In [123]:
def clean_numeric_columns(df, columns):
    for col in columns:
        # Replace known junk markers
        df[col] = df[col].replace(
            ["?", "**", "nan", "NaN", "NA", "N/A", "", " "],
            np.nan
        )

        # Strip whitespace
        df[col] = df[col].astype(str).str.strip()

        # Convert to numeric
        df[col] = pd.to_numeric(df[col], errors="coerce")

        # Fill missing values with median
        df[col].fillna(df[col].median(), inplace=True)

    return df
df = clean_numeric_columns(df, ['pclass', 'survived', 'age', 'family', 'fare'])

C:\Users\dhruv joshi\AppData\Local\Temp\ipykernel_19228\2771433854.py:16: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df[col].fillna(df[col].median(), inplace=True)
C:\Users\dhruv joshi\AppData\Local\Temp\ipykernel_19228\2771433854.py:16: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a co

In [124]:
def clean_non_numeric_columns(df):
    text_cols = df.select_dtypes(include=["object"]).columns

    for col in text_cols:
        df[col] = (
            df[col]
            .astype(str)
            .str.strip()
            .str.replace(r"[^\x00-\x7F]+", " ", regex=True)
            .str.replace(r"\s+", " ", regex=True)
            .replace(["?", "**", "nan", "NaN", "NA", "N/A", "", " "], np.nan)
        )

    return df
df = clean_non_numeric_columns(df)

In [125]:
# --------------------------------
# Encode categorical features
# --------------------------------
le = LabelEncoder()
df["gender"] = le.fit_transform(df["gender"])
df["embarked"] = le.fit_transform(df["embarked"])

In [126]:
# --------------------------------
# Split features and target
# --------------------------------
X = df.drop("survived", axis=1)
y = df["survived"]



In [127]:
# --------------------------------
# Feature Selection
# --------------------------------
scaler = MinMaxScaler()
X_scaled = scaler.fit_transform(X)

chi_scores, _ = chi2(X_scaled, y)
chi_rank = pd.Series(chi_scores, index=X.columns).sort_values(ascending=False)

ig_scores = mutual_info_classif(X, y, random_state=41)
ig_rank = pd.Series(ig_scores, index=X.columns).sort_values(ascending=False)

print("Chi-Square Ranking:")
print(chi_rank)
print("\nInformation Gain Ranking:")
print(ig_rank)


Chi-Square Ranking:
gender      65.177345
pclass      33.824184
fare        10.136493
embarked     6.086016
family       0.223519
age          0.118664
dtype: float64

Information Gain Ranking:
gender      0.158684
fare        0.125163
family      0.069000
pclass      0.057189
age         0.018595
embarked    0.000000
dtype: float64


In [128]:
# --------------------------------
# Baseline Model
# --------------------------------
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42
)

clf = LogisticRegression(max_iter=3000)
clf.fit(X_train, y_train)
baseline_acc = accuracy_score(y_test, clf.predict(X_test))


In [129]:
# --------------------------------
# PCA
# --------------------------------
pca = PCA(n_components=4, random_state=42)
X_pca = pca.fit_transform(X)

Xp_train, Xp_test, yp_train, yp_test = train_test_split(
    X_pca, y, test_size=0.3, random_state=42
)

clf.fit(Xp_train, yp_train)
pca_acc = accuracy_score(yp_test, clf.predict(Xp_test))

In [130]:
# --------------------------------
# t-SNE
# --------------------------------
tsne = TSNE(
    n_components=2,
    perplexity=30,
    learning_rate=200,
    random_state=42
)

X_tsne = tsne.fit_transform(X)

Xt_train, Xt_test, yt_train, yt_test = train_test_split(
    X_tsne, y, test_size=0.3, random_state=42
)

clf.fit(Xt_train, yt_train)
tsne_acc = accuracy_score(yt_test, clf.predict(Xt_test))


In [131]:
# --------------------------------
# Results
# --------------------------------
print("\nAccuracy Comparison")
print("Baseline Accuracy:", baseline_acc)
print("PCA Accuracy:", pca_acc)
print("t-SNE Accuracy:", tsne_acc)



Accuracy Comparison
Baseline Accuracy: 0.8081841432225064
PCA Accuracy: 0.6547314578005116
t-SNE Accuracy: 0.629156010230179


**The baseline model trained on the full feature set achieved the highest accuracy of 80.8 percent, indicating that the original features contain strong discriminative information for survival prediction. Applying PCA resulted in a significant drop in accuracy to 65.5 percent because PCA preserves variance rather than class separability, leading to the removal of features that are important for classification despite having lower variance. The performance degraded further with t-SNE, which achieved an accuracy of 62.9 percent. This outcome is expected, as t-SNE is designed for visualization and local structure preservation rather than for downstream predictive modeling, and it does not maintain the global relationships required for reliable classification. Overall, dimensionality reduction did not improve model performance for this dataset and instead led to information loss.**